# télos Unified 3-Paradigm Training Suite
This notebook executes sequential training for the exact same model scale and token ratio across all three paradigms:
1. **AR Baseline** (Causal Attention, Next-Token Prediction)
2. **MDLM** (Bidirectional Attention, Absorbing Masked Discrete Diffusion)
3. **UNDLM** (Bidirectional Attention, Reversible Uniform Noise Diffusion)

All three runs use identical hyperparameters, model parameter counts, token budgets, and data ordering for a strict 1-to-1 controlled comparison.

In [ ]:
import os
import sys
import time
import gc
import yaml
from pathlib import Path

# Ensure working directory is project root
project_root = Path.cwd()
while not (project_root / "mdiff").exists() and project_root.parent != project_root:
    project_root = project_root.parent
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
from mdiff.model.mlx_components import MLXTelosTransformer
from mdiff.training.trainer import TelosMLXTrainer as MDLM_Trainer
from undiff.training.trainer import TelosMLXUNDLMTrainer as UNDLM_Trainer
from ar.model.mlx_components import MLXCausalTransformer
from ar.training.trainer import TelosMLXARTrainer as AR_Trainer

def run_unified_training_suite(config_path):
    """Runs 3-paradigm controlled training (AR, MDLM, UNDLM) on identical configurations."""
    with open(config_path, "r") as f:
        base_cfg = yaml.safe_load(f)
    
    stem = Path(config_path).stem
    tier = "25m" if "25m" in stem else ("50m" if "50m" in stem else "12m")
    save_every = base_cfg.get("checkpoint", {}).get("save_every_steps", 25)
    
    suite_start = time.time()
    print("=" * 90)
    print(f"STARTING UNIFIED 3-PARADIGM SUITE FOR: {stem} (Tier: {tier})")
    print(f"Max Steps: {base_cfg.get('training', {}).get('max_steps')} | Batch Size: {base_cfg.get('training', {}).get('batch_size')} | Grad Accum: {base_cfg.get('training', {}).get('gradient_accumulation')}")
    print("=" * 90)
    
    # ---------------------------------------------------------
    # PARADIGM 1: AR Baseline
    # ---------------------------------------------------------
    print("\n>>> PARADIGM 1/3: Autoregressive (AR) Baseline <<<")
    ar_cfg = yaml.safe_load(yaml.dump(base_cfg))
    ar_cfg["checkpoint"] = {"dir": f"checkpoints/ar/{tier}/{stem}", "save_every_steps": save_every}
    
    ar_model = MLXCausalTransformer(**ar_cfg["model"])
    ar_model.set_dtype(mx.bfloat16)
    ar_trainer = AR_Trainer(ar_model, ar_cfg)
    ar_trainer.train()
    del ar_model, ar_trainer
    gc.collect()
    mx.clear_cache()
    
    # ---------------------------------------------------------
    # PARADIGM 2: Masked Diffusion (MDLM)
    # ---------------------------------------------------------
    print("\n>>> PARADIGM 2/3: Masked Discrete Diffusion (MDLM) <<<")
    mdlm_cfg = yaml.safe_load(yaml.dump(base_cfg))
    mdlm_cfg["checkpoint"] = {"dir": f"checkpoints/masked/{tier}/{stem}", "save_every_steps": save_every}
    
    mdlm_model = MLXTelosTransformer(**mdlm_cfg["model"])
    mdlm_model.set_dtype(mx.bfloat16)
    mdlm_trainer = MDLM_Trainer(mdlm_model, mdlm_cfg)
    mdlm_trainer.train()
    del mdlm_model, mdlm_trainer
    gc.collect()
    mx.clear_cache()
    
    # ---------------------------------------------------------
    # PARADIGM 3: Uniform Noise Diffusion (UNDLM)
    # ---------------------------------------------------------
    print("\n>>> PARADIGM 3/3: Uniform Noise Diffusion (UNDLM) <<<")
    undlm_cfg = yaml.safe_load(yaml.dump(base_cfg))
    undlm_cfg["checkpoint"] = {"dir": f"checkpoints/uniform/{tier}/{stem}", "save_every_steps": save_every}
    
    undlm_model = MLXTelosTransformer(**undlm_cfg["model"])
    undlm_model.set_dtype(mx.bfloat16)
    undlm_trainer = UNDLM_Trainer(undlm_model, undlm_cfg)
    undlm_trainer.train()
    del undlm_model, undlm_trainer
    gc.collect()
    mx.clear_cache()
    
    total_hours = (time.time() - suite_start) / 3600.0
    print("=" * 90)
    print(f"ALL 3 PARADIGMS COMPLETED SUCCESSFULLY IN {total_hours:.2f} HOURS!")
    print("=" * 90)


In [ ]:
# EXECUTE UNIFIED SUITE: 12.5M 1:1 Ratio (~12.5M tokens, 96 steps)
run_unified_training_suite("configs/unified/12m/telos_12m_r1.yaml")

# Uncomment for subsequent ratios in 12.5M scale study:
# run_unified_training_suite("configs/unified/12m/telos_12m_r5.yaml")
# run_unified_training_suite("configs/unified/12m/telos_12m_r10.yaml")
# run_unified_training_suite("configs/unified/12m/telos_12m_r15.yaml")
# run_unified_training_suite("configs/unified/12m/telos_12m_r20.yaml")
# run_unified_training_suite("configs/unified/12m/telos_12m_r25.yaml")
# run_unified_training_suite("configs/unified/12m/telos_12m_r30.yaml")
